<a href="https://colab.research.google.com/github/ECEZHAO2013/Building-LLM-From-Scratch/blob/main/notebooks/1_LLM_training_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## download data
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x796d588a0dd0>)

In [2]:
## reading a short story
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [6]:
## demo with re
import re
text = "Hello, world. This, is a test"
results = re.split(r'(\s)', text) ## () will keep the space in the results; removing () will only keep words
print(results)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test']


In [7]:
## adding more delimiters
results = re.split(r"([, .]|\s)", text)
print(results)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test']


In [8]:
results = [item for item in results if item.strip()]
print(results)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test']


In [12]:
## modify it a bit further
text = "Hello, world. is this-- a test?"
results = re.split(r'([.,:;?!"()\']|--|\s)', text)
results = [item.strip() for item in results if item.strip()] ## remove leading and trailing space
## results = [item for item in results if item.strip()]   ## will not removing leading and trailing space
print(results)

['Hello', ',', 'world', '.', 'is', 'this', '--', 'a', 'test', '?']


In [15]:
## apply to the text we downloaded
preprocessed = re.split(r'([.,:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [16]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [17]:
## Create a vocabulary and token IDs by AI assistant Gemini 2.5
vocabulary = sorted(list(set(preprocessed)))
token_to_int = {token: i for i, token in enumerate(vocabulary)}
int_to_token = {i: token for i, token in enumerate(vocabulary)}

token_ids = [token_to_int[token] for token in preprocessed]

print(f"Vocabulary size: {len(vocabulary)}")
print(f"First 10 tokens: {preprocessed[:10]}")
print(f"First 10 token IDs: {token_ids[:10]}")

Vocabulary size: 1130
First 10 tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']
First 10 token IDs: [53, 44, 149, 1003, 57, 38, 818, 115, 256, 486]


In [19]:
## manually create vocab
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [21]:
## creating vocabulary and print its first 51 entries
vocab = {token: i for i, token in enumerate(all_words)}
#print(vocab)
for i, item in enumerate(vocab.items()):
  if i < 51:     ## use if, the code will check all vocab and only print the first 50 vocab
    print(item)

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [22]:
for i, item in enumerate(vocab.items()):
  print(item)
  if i >= 50:
    break ## the code only check and print the first 50 vocab and stop;  faster

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [32]:
## TOKENizer Class
class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int = vocab ## string int pair
    self.int_to_str = {i:s for s, i in vocab.items()} ## convert it back to string from int

  def encode(self, text): ## process string to ids
    preprocessed = re.split(r'([.,:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):  ## convert token ids back into text
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([.,?!"()\'])', r'\1', text) ## remove space before specific punc
    return text

In [33]:
## example
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [34]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [35]:
## example not in the text: Hello is not a word in the vocab text so getting error here
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'